In [ ]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.split import make_time_folds, expanding_window_splits
from src.features.build_features import build_features

DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PREDICTIONS = PROJECT_ROOT / "data" / "predictions"
REPORTS_FIGURES = PROJECT_ROOT / "reports" / "figures"

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.figsize"] = (10, 5)

# Загружаем
train = pd.read_parquet(DATA_INTERIM / "train.parquet", engine="pyarrow")
test = pd.read_parquet(DATA_INTERIM / "test.parquet", engine="pyarrow")

# Патч колонок в test
rename_map = {c: c.replace("id-", "id_") for c in test.columns if c.startswith("id-")}
test = test.rename(columns=rename_map)

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")

In [ ]:
folds = make_time_folds(train["TransactionDT"], n_splits=5)
target = train["isFraud"].astype(np.int8)

# Feature engineering с UID
t0 = time.time()
train_fe, test_fe = build_features(
    train=train.copy(),
    test=test.copy(),
    target=target,
    folds=folds,
)
print(f"\nFE total time: {time.time() - t0:.1f}s")
print(f"Train FE: {train_fe.shape}")
print(f"Test FE:  {test_fe.shape}")

In [ ]:
folds = make_time_folds(train["TransactionDT"], n_splits=5)
target = train["isFraud"].astype(np.int8)

# Feature engineering с UID
t0 = time.time()
train_fe, test_fe = build_features(
    train=train.copy(),
    test=test.copy(),
    target=target,
    folds=folds,
)
print(f"\nFE total time: {time.time() - t0:.1f}s")
print(f"Train FE: {train_fe.shape}")
print(f"Test FE:  {test_fe.shape}")

In [ ]:
uid_cols = [c for c in train_fe.columns if "uid" in c.lower() or c == "D1n"]
print(f"UID-related columns ({len(uid_cols)}):")
for c in uid_cols:
    n_null = train_fe[c].isna().sum()
    pct_null = n_null / len(train_fe) * 100
    dtype = train_fe[c].dtype
    print(f"  {c:35s}  dtype={str(dtype):12s}  nulls={n_null:>7,} ({pct_null:.1f}%)")

print()
print(f"UID unique values: {train_fe['uid'].nunique():,}")
print(f"UID coverage (non-null): {(1 - train_fe['uid'].isna().mean()) * 100:.1f}%")

In [ ]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

# Готовим X / y
# Исключаем сам uid (строка) — LightGBM не проглотит, а cat-версии и так закодированы через uid_freq/uid_te
drop_cols = ["TransactionID", "isFraud", "TransactionDT", "uid"]
y = train_fe["isFraud"].astype(np.int8)
X = train_fe.drop(columns=drop_cols)

cat_cols = X.select_dtypes(include=["category"]).columns.tolist()
print(f"X: {X.shape}, cat cols: {len(cat_cols)}")

lgb_params = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_child_samples": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "n_jobs": -1,
    "random_state": 42,
}

splits = expanding_window_splits(folds, n_splits=5)
oof_preds = np.zeros(len(X), dtype=np.float32)
cv_scores = []
models = []

for fold_num, (train_idx, valid_idx) in enumerate(splits, start=1):
    t0 = time.time()
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[valid_idx], y.iloc[valid_idx]

    train_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols)
    valid_data = lgb.Dataset(X_va, label=y_va, categorical_feature=cat_cols,
                              reference=train_data)

    model = lgb.train(
        params=lgb_params,
        train_set=train_data,
        num_boost_round=2000,
        valid_sets=[valid_data],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )

    oof_preds[valid_idx] = model.predict(X_va, num_iteration=model.best_iteration)
    fold_auc = roc_auc_score(y_va, oof_preds[valid_idx])
    cv_scores.append(fold_auc)
    models.append(model)

    elapsed = time.time() - t0
    print(f"Fold {fold_num}: AUC = {fold_auc:.5f} "
          f"(best_iter={model.best_iteration}, time={elapsed:.0f}s)")

print()
print(f"CV AUC: {np.mean(cv_scores):.5f} ± {np.std(cv_scores):.5f}")
valid_mask = folds > 0
print(f"OOF AUC: {roc_auc_score(y[valid_mask], oof_preds[valid_mask]):.5f}")

# История
print()
print(f"v0 (baseline):       0.90742")
print(f"v1 (+time/money/agg): 0.90945 (+0.002)")
print(f"v2 (+encodings):     0.90865 (-0.001)")
print(f"v3 (+UID):           {np.mean(cv_scores):.5f} "
      f"(+{np.mean(cv_scores) - 0.90865:+.5f} vs v2)")

In [ ]:
X_test = test_fe.drop(columns=["TransactionID", "TransactionDT", "uid"])
X_test = X_test[X.columns]

for col in cat_cols:
    if col in X_test.columns:
        X_test[col] = X_test[col].astype(X[col].dtype)

test_preds = np.zeros(len(X_test), dtype=np.float32)
for i, model in enumerate(models):
    p = model.predict(X_test, num_iteration=model.best_iteration)
    test_preds += p / len(models)

print(f"Predictions: min={test_preds.min():.4f}, max={test_preds.max():.4f}, "
      f"mean={test_preds.mean():.4f}")

submission = pd.DataFrame({
    "TransactionID": test_fe["TransactionID"],
    "isFraud": test_preds,
})
submission.to_csv(DATA_PREDICTIONS / "sub_lgbm_uid_v3.csv", index=False)

oof_df = pd.DataFrame({
    "TransactionID": train_fe["TransactionID"],
    "isFraud_true": y,
    "isFraud_pred": oof_preds,
    "fold": folds,
})
oof_df.to_csv(DATA_PREDICTIONS / "oof_lgbm_uid_v3.csv", index=False)

print("Saved: sub_lgbm_uid_v3.csv + oof_lgbm_uid_v3.csv")